# 02 · Thread reconstruction
Walk `in_response_to_tweet_id` chains into (customer message → brand reply) pairs.

The production logic lives in `src/build_pairs.py`; this notebook shows *how it works* and inspects the output. Run `python -m src.build_pairs` from repo root first (or run the cell below).

In [1]:
import os, sys
os.environ['PYTHONUTF8'] = '1'
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')  # run from repo root so relative data paths resolve
import pandas as pd
from src import build_pairs
if not os.path.exists('data/uber_pairs.csv'):
    build_pairs.main()

## How a pair is formed
For each Uber_Support outbound reply, look up its `in_response_to_tweet_id`; if that parent is an inbound customer tweet, we have one (customer_msg → brand_reply) pair. We keep the first brand reply per customer tweet.

In [2]:
pairs = pd.read_csv('data/uber_pairs.csv')
pool = pd.read_csv('data/uber_golden_pool.csv')
print('grounding pairs:', len(pairs), '| held-out pool:', len(pool))
print('disjoint?', set(pairs['pair_id']).isdisjoint(set(pool['pair_id'])))
pairs.head(3)

grounding pairs: 1500 | held-out pool: 600
disjoint? True


,pair_id,customer_tweet_id,customer_msg,brand_reply,created_at
0,uc_875043,875043,When you order @115877 instead of @2350 and Ub...,@327746 Sorry to hear about this. A member of ...,Fri Oct 20 20:44:58 +0000 2017
1,uc_2256096,2256096,"Are you kidding, @115873?! All the routes your...",@657100 Here to help. Send us a note at https:...,Fri Nov 10 22:23:04 +0000 2017
2,uc_1836147,1836147,@Uber_Support That link didn’t take me anywher...,"@550583 Sorry for the trouble, Sasha. Please D...",Wed Oct 18 15:34:02 +0000 2017


## Inspect reconstructed pairs

In [3]:
for _, r in pairs.sample(6, random_state=2).iterrows():
    print('CUSTOMER:', r['customer_msg'][:130])
    print('BRAND   :', r['brand_reply'][:130])
    print()

CUSTOMER: @115873 THIS MEATBALL CHARGED ME $50 FOR A RIDE THAT HE NEVER PICKED ME UP ON!!!
BRAND   : @347412 We're sorry to hear this was your experience. Contact us via https://t.co/e8HVorX5rh and we'll be in touch.

CUSTOMER: @Uber_Support Hello. The outreach team has not read my inquiry because they keep sending me links to a page to a form that I have 
BRAND   : @425873 We understand that you would like to speak on the phone about this concern. At this time, we are only able to offer suppor

CUSTOMER: @Uber_Support moved to Qatar from India still receiving promos for India nd not for Qatar. Please help!!!
BRAND   : @763396 Happy to help! Please reach out to https://t.co/QDn91HzZV0 and fill out the 4 boxes at the bottom so we can get in touch.

CUSTOMER: Hey @Uber_Support I think there's a glitch in your notifications. I live in downtown DC, so this makes little sense 😉 https://t.co
BRAND   : @778590 We apologize for any inconvenience. Kindly send a note here: https://t.co/e8HVorX5r

## The 'please DM us' problem is visible already

In [4]:
from src import pipeline
dm_rate = pairs['brand_reply'].apply(pipeline.is_dm_deflection).mean()
print(f'{dm_rate*100:.0f}% of brand replies are pure DM-deflection templates.')
print('=> any model that grounds in these inherits the laziness. Tracked in eval + failure analysis.')

61% of brand replies are pure DM-deflection templates.
=> any model that grounds in these inherits the laziness. Tracked in eval + failure analysis.
